# 02 – Comparação dos resultados dos modelos

Lê os `metricas.json` gravados em `models/<modelo>/<cenário>/` pelos scripts de `src/models/` e organiza as métricas para consulta rápida. As tabelas e figuras finais do TCC, com IC 95% e teste de DeLong, são geradas por `src/visualization/gerar_resultados.py`.

Execute a partir da pasta `ia/notebooks`, depois de treinar os modelos (`python main.py`).

In [ ]:
import json
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent
linhas = []
for arq in sorted((RAIZ / "models").glob("*/*/metricas.json")):
    m = json.loads(arq.read_text(encoding="utf-8"))
    linhas.append({"modelo": arq.parent.parent.name, "cenario": arq.parent.name,
                   "balanceamento": m["balanceamento"],
                   "AUC validação": m["auc_validacao_cruzada"], **m["teste"]})
resultados = pd.DataFrame(linhas)
resultados.round(4)

## AUC no teste: só individuais × individuais + municipais

In [ ]:
auc = resultados.pivot(index="modelo", columns="cenario", values="AUC")
auc["ganho"] = auc["individuais_municipais"] - auc["individuais"]
auc.sort_values("individuais_municipais", ascending=False).round(4)

## Hiperparâmetros escolhidos

In [ ]:
for arq in sorted((RAIZ / "models").glob("*/*/metricas.json")):
    m = json.loads(arq.read_text(encoding="utf-8"))
    print(f"{arq.parent.parent.name:22} {arq.parent.name:24} {m['balanceamento']:18} {m['hiperparametros']}")

## Importância por permutação (cenário com municipais)

In [ ]:
importancia = {}
for arq in sorted((RAIZ / "models").glob("*/individuais_municipais/importancia_permutacao.csv")):
    importancia[arq.parent.parent.name] = pd.read_csv(arq, sep=";").set_index("variavel").iloc[:, 0]
importancia = pd.DataFrame(importancia)
importancia.loc[importancia.mean(axis=1).sort_values(ascending=False).index].round(4)